In [ ]:
from datasets import load_dataset

dataset = load_dataset("pavanmantha/medical-symptoms-disease-classification")

print(dataset)

In [ ]:
import pandas as pd

df = pd.DataFrame(dataset['train'])
df.head()

In [ ]:
def format_prompts_train(examples):
  instruction = "Classify the sicknes based on the following description: "
  inputs = examples['text']
  outputs = examples['label']

  texts = []
  for i, o in zip(inputs, outputs):
    text = f"{instruction}\n Input: {i}\n Output: {o}"
    texts.append(text)
  return {"text": texts}

dataset_train = dataset['train'].map(
    format_prompts_train,
    batched = True,
    remove_columns = dataset['train'].column_names
)

In [ ]:
dataset_train

In [ ]:
df_train = pd.DataFrame(dataset_train)

df_train.head()

In [ ]:
def format_prompts_test(examples):
  instruction = "Classify the sicknes based on the following description: "
  inputs = examples['text']
  outputs = examples['label']

  texts = []
  for i, o in zip(inputs, outputs):
    text = f"{instruction}\n Input: {i}\n Output: "
    texts.append(text)
  return {"text": texts}

dataset_test = dataset['test'].map(format_prompts_test, batched = True)

df_test = pd.DataFrame(dataset_test)
df_test.head()

# Baseline

In [ ]:
from unsloth import FastLanguageModel
import torch

model_base, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-v0.3-bnb-4bit",
    max_seq_length = 512,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model_base)

In [ ]:
def batch_inference_base(dataset_segment, batch_size=16):
    all_preds = []
    for i in range(0, len(dataset_segment), batch_size):
        batch = dataset_segment[i : i + batch_size]
        inputs = tokenizer(batch["text"], return_tensors="pt", padding=True).to("cuda")

        outputs = model_base.generate(
            **inputs,
            max_new_tokens=15,
            temperature=0.1,
            repetition_penalty=1.2
        )

        input_len = inputs.input_ids.shape[1]
        decoded = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)
        all_preds.extend([res.strip().split('\n')[0] for res in decoded])
    return all_preds

predictions_base = batch_inference_base(dataset_test, batch_size=16)

labels_reais = dataset['test']['label']

from sklearn.metrics import classification_report
print(classification_report(labels_reais, predictions_base))

# Fine-tuning with QLoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/mistral-7b-v0.3-bnb-4bit',
    load_in_4bit=True
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth'
)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_train,
    dataset_text_field = "text",
    max_seq_length = 128,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        max_steps = 60,
        num_train_epochs=2,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

In [ ]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
trainer_stats = trainer.train()

In [ ]:
import matplotlib.pyplot as plt

history = trainer_stats.metrics
steps = [x['step'] for x in trainer.state.log_history if 'loss' in x]
loss = [x['loss'] for x in trainer.state.log_history if 'loss' in x]

plt.figure(figsize=(10, 6))
plt.plot(steps, loss, label='Training Loss', color='royalblue', linewidth=2)

plt.title('Training Loss Schedule', fontsize=14)
plt.xlabel('Steps', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.savefig("loss_chart.png", dpi=300)
plt.show()

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

In [ ]:
import torch
from sklearn.metrics import classification_report

labels_reais = dataset['test']['label']

tokenizer.padding_side = "left"

def batch_inference(dataset_segment, batch_size=16):
    all_preds = []
    for i in range(0, len(dataset_segment), batch_size):
        batch = dataset_segment[i : i + batch_size]
        inputs = tokenizer(batch["text"], return_tensors="pt", padding=True).to("cuda")

        outputs = model.generate(
            **inputs,
            max_new_tokens=15,
            temperature=0.1,
            use_cache=True,
            repetition_penalty=1.2
        )

        input_len = inputs.input_ids.shape[1]
        decoded = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)
        all_preds.extend([res.strip().split('\n')[0] for res in decoded])

    return all_preds

predictions = batch_inference(dataset_test, batch_size=16)

In [ ]:
from difflib import get_close_matches

labels_validas = list(set(dataset['train']['label']))

def normalizar_predicao(pred):
    match = get_close_matches(pred, labels_validas, n=1, cutoff=0.3)
    return match[0] if match else pred

predictions_normalizadas = [normalizar_predicao(p) for p in predictions]
print(classification_report(labels_reais, predictions_normalizadas))

In [ ]:
model.save_pretrained("fine-tuning-mistral-7b-medical")
tokenizer.save_pretrained("fine-tuning-mistral-7b-medical")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

classes = sorted(list(set(labels_reais)))

cm = confusion_matrix(labels_reais, predictions, labels=classes)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)

plt.title('Confusion Matrix - Medical Diagnosis Model')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.savefig("confusion_matrix.png", dpi=300)
plt.show()

#### The model frequently misclassifies Pneumonia as Bronchial Asthma. This is attributed to the symptomatic overlap in features like cough and dyspnea. Improving the model would require specific clinical markers such as 'High Fever' or 'Wheezing' to be more prevalent in the training prompts

# Inference with interface

In [ ]:
def predict_desiease(symptoms_text):
    prompt = f"Classify the sicknes based on the following description:\n Input:\n{symptoms_text}\n Output:\n"

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=15,
        temperature=0.1,
        use_cache=True,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    input_len = inputs.input_ids.shape[1]
    decoded = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)

    return decoded[0].strip().split('\n')[0]


In [ ]:
import gradio as gr

def predict_ui(sintomas):
    if not sintomas.strip():
        return "Insert the symptoms for analysis."

    resultado = predict_desiease(sintomas)
    return resultado

demo = gr.Interface(
    fn=predict_ui,
    inputs=gr.Textbox(lines=5, label="Discribe the symptoms", placeholder="Ex: fever, cough, joint pain..."),
    outputs=gr.Textbox(label="Probable diagnosis"),
    title="AI Medical Classifier (Mistral-7B QLoRA)",
    description="Testing interface for a desiease classification model trained via Fine-tuning with QLoRA.",
    theme="soft"
)

demo.launch(share=True) # O 'share=True' cria o link público na hora!